#Read csv file using data frame reader API

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config



In [0]:
%run ../00-common/02.BronzeHelper

In [0]:
source_path=f"{landing_folder_path}/{v_batch_id}/constructors.json"
table_name=f"{catalog_name}.{bronze_schema}.constructors"

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *

# This scheam is also correct and working
# constructor_schema = StructType([
#   StructField('constructorId', StringType(), True),
#   StructField('name', StringType(), True),
#    StructField('nationality', StringType(), True),
#   StructField('url', StringType(), True)
# ])
constructor_schema = """constructorId STRING, 
                        name STRING, nationality STRING, url STRING"""

df_constructor = (spark.read.format("json")
               .option('header', True)
               .option('mode', 'FAILFAST')  # strict mode datatype validation
             #  .option('mod', 'PERMISSIVE') # ignore bad records with null value
              # .option('inferSchema', True)  optional incase of schema passing as below
               .schema(constructor_schema)
               .load(source_path))

In [0]:
df_constructor.show();

In [0]:
import pyspark.sql.functions as F

df_constructor_final=add_ingestion_metadata(df_constructor)
display(df_constructor_final)

In [0]:
# (df_constructor_final.write
#       .format("delta")
#       .mode("overwrite")
#       .saveAsTable(table_name))

In [0]:
write_to_bronze(input_df=df_constructor_final,
                target_table=table_name,
                batch_id=v_batch_id)

In [0]:
%sql
select * from formula1_incr_catalog.bronze.constructors

In [0]:
df_table=spark.read.table(table_name)
display(df_table)